# Principal component analysis


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
RESULTS_ROOT = Path(os.environ.get('EMOTION_RESULTS_ROOT', 'results'))
INPUT_FILE = Path(
    os.environ.get(
        'EMOTION_MAIN_SUMMARY_FILE',
        str(
            RESULTS_ROOT / '1_reliability_description' / 'results_files' / 'reliability_description_summary.xlsx'
        )
    )
)
OUTPUT_DIR = Path(
    os.environ.get('EMOTION_PCA_OUTPUT_DIR', str(RESULTS_ROOT / '2_principal_component_analysis'))
)
PROCESS_DIR = OUTPUT_DIR / 'process_files'
RESULTS_DIR = OUTPUT_DIR / 'results_files'
for folder in [OUTPUT_DIR, PROCESS_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
N_PERMUTATIONS = int(os.environ.get('EMOTION_N_PERMUTATIONS', '10000'))
PARALLEL_PERCENTILE = 95
PARALLEL_RANDOM_SEED = 2026
N_TOP_LOADINGS = 10
N_EXTREME_WORDS = 5
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Cannot find input file: {INPUT_FILE}')


In [ ]:
def z_standardize_columns(data):
    array = np.asarray(data, dtype=float)
    means = array.mean(axis=0)
    sds = array.std(axis=0, ddof=1)
    if np.any(~np.isfinite(sds)) or np.any(sds <= 0):
        raise ValueError('At least one feature has a zero or invalid sample SD.')
    return ((array - means) / sds, means, sds)

def fit_pca(standardized_data, n_components):
    model = PCA(n_components=n_components, svd_solver='full')
    scores = model.fit_transform(standardized_data)
    eigenvalues = model.explained_variance_.copy()
    ratios = model.explained_variance_ratio_.copy()
    loadings = model.components_.T * np.sqrt(eigenvalues)
    return (model, scores, loadings, eigenvalues, ratios)

def orient_by_largest_loading(loadings, scores=None, feature_names=None):
    oriented_loadings = np.asarray(loadings, dtype=float).copy()
    oriented_scores = None if scores is None else np.asarray(scores, dtype=float).copy()
    rows = []
    for component_index in range(oriented_loadings.shape[1]):
        anchor_index = int(np.argmax(np.abs(oriented_loadings[:, component_index])))
        before = float(oriented_loadings[anchor_index, component_index])
        sign = 1.0 if before >= 0 else -1.0
        oriented_loadings[:, component_index] *= sign
        if oriented_scores is not None:
            oriented_scores[:, component_index] *= sign
        rows.append(
            {
                'Component': f'PC{component_index + 1}',
                'Anchor Feature': feature_names[anchor_index] if feature_names is not None else anchor_index,
                'Loading Before Orientation': before,
                'Sign Multiplier': sign,
                'Loading After Orientation': float(oriented_loadings[anchor_index, component_index])
            }
        )
    return (oriented_loadings, oriented_scores, pd.DataFrame(rows))

def parallel_analysis(standardized_data, n_permutations, percentile, seed):
    X = np.asarray(standardized_data, dtype=float)
    n_rows, n_features = X.shape
    n_components = min(n_rows, n_features)
    observed = np.linalg.eigvalsh(np.corrcoef(X, rowvar=False))[::-1]
    rng = np.random.default_rng(seed)
    permuted = np.empty_like(X)
    null_eigenvalues = np.empty((n_permutations, n_components), dtype=float)
    for iteration in range(n_permutations):
        for feature_index in range(n_features):
            permuted[:, feature_index] = rng.permutation(X[:, feature_index])
        null_eigenvalues[iteration] = np.linalg.eigvalsh(np.corrcoef(permuted, rowvar=False))[::-1]
    null_threshold = np.percentile(null_eigenvalues, percentile, axis=0)
    above = observed > null_threshold
    retained = 0
    for flag in above:
        if bool(flag):
            retained += 1
        else:
            break
    labels = [f'PC{index}' for index in range(1, n_components + 1)]
    summary = pd.DataFrame(
        {
            'Component': labels,
            'Observed Eigenvalue': observed,
            'Null Mean': null_eigenvalues.mean(axis=0),
            'Null SD': null_eigenvalues.std(axis=0, ddof=1),
            'Null 2.5th Percentile': np.percentile(null_eigenvalues, 2.5, axis=0),
            'Null Median': np.percentile(null_eigenvalues, 50, axis=0),
            f'Null {percentile}th Percentile': null_threshold,
            'Null 97.5th Percentile': np.percentile(null_eigenvalues, 97.5, axis=0),
            f'Observed Minus Null {percentile}th': observed - null_threshold,
            f'Observed Above Null {percentile}th': above,
            'Retained Sequentially': [index < retained for index in range(n_components)]
        }
    )
    return (summary, null_eigenvalues, retained)

def make_top_loading_table(loadings_frame, n_top=10):
    rows = []
    for component in loadings_frame.columns:
        series = loadings_frame[component]
        order = series.abs().sort_values(ascending=False).index
        for rank, feature in enumerate(order[:n_top], start=1):
            value = float(series.loc[feature])
            rows.append(
                {
                    'Component': component,
                    'Rank by Absolute Loading': rank,
                    'Emotion Feature': feature,
                    'Loading': value,
                    'Absolute Loading': abs(value)
                }
            )
    return pd.DataFrame(rows)

def make_extreme_word_table(scores_frame, n_extreme=5):
    rows = []
    for component in scores_frame.columns:
        series = scores_frame[component]
        for direction, values in [
            ('Lowest', series.sort_values().head(n_extreme)),
            ('Highest', series.sort_values(ascending=False).head(n_extreme))
        ]:
            for rank, (word, value) in enumerate(values.items(), start=1):
                rows.append(
                    {
                        'Component': component,
                        'Direction': direction,
                        'Rank Within Direction': rank,
                        'Emotion Word': word,
                        'Component Score': float(value)
                    }
                )
    return pd.DataFrame(rows)


In [ ]:
mean_matrix_raw = pd.read_excel(INPUT_FILE, sheet_name='Mean_Rating_Matrix')
if 'Emotion Word' not in mean_matrix_raw.columns:
    raise ValueError("Mean_Rating_Matrix must contain an 'Emotion Word' column.")
emotion_words = mean_matrix_raw['Emotion Word'].astype('string').str.strip()
feature_names = [column for column in mean_matrix_raw.columns if column != 'Emotion Word']
mean_matrix = mean_matrix_raw[feature_names].apply(pd.to_numeric, errors='coerce')
if len(emotion_words) != 70 or len(feature_names) != 43:
    raise ValueError(
        f'Expected a 70 x 43 matrix, found {len(emotion_words)} x {len(feature_names)}.'
    )
if emotion_words.isna().any() or emotion_words.duplicated().any():
    raise ValueError('Emotion words contain missing or duplicate values.')
if mean_matrix.isna().any().any():
    raise ValueError('The mean-rating matrix contains missing or nonnumeric values.')
if not mean_matrix.apply(lambda column: column.between(1, 9)).all().all():
    raise ValueError('Some mean ratings fall outside the 1–9 scale.')
X_raw = mean_matrix.to_numpy(dtype=float)
X, feature_means, feature_sds = z_standardize_columns(X_raw)
n_words, n_features = X.shape
data_qc = pd.DataFrame(
    {
        'Check': [
            'Number of Emotion Words',
            'Number of Emotion Features',
            'Missing Matrix Cells',
            'Duplicate Emotion Words',
            'Minimum Mean Rating',
            'Maximum Mean Rating',
            'Maximum Absolute Standardized Mean',
            'Maximum Absolute Standardized-SD Deviation from 1'
        ],
        'Result': [
            n_words,
            n_features,
            int(mean_matrix.isna().sum().sum()),
            int(emotion_words.duplicated().sum()),
            float(mean_matrix.min().min()),
            float(mean_matrix.max().max()),
            float(np.max(np.abs(X.mean(axis=0)))),
            float(np.max(np.abs(X.std(axis=0, ddof=1) - 1)))
        ]
    }
)
standardization_summary = pd.DataFrame(
    {
        'Emotion Feature': feature_names,
        'Original Mean': feature_means,
        'Original SD': feature_sds,
        'Standardized Mean': X.mean(axis=0),
        'Standardized SD': X.std(axis=0, ddof=1)
    }
)
pd.concat([emotion_words.rename('Emotion Word'), mean_matrix], axis=1).to_csv(
    PROCESS_DIR / '01_mean_rating_matrix_input.csv',
    index=False,
    encoding='utf-8-sig'
)
pd.DataFrame(X, columns=feature_names).assign(**{'Emotion Word': emotion_words}).loc[:, ['Emotion Word'] + feature_names].to_csv(
    PROCESS_DIR / '02_standardized_mean_rating_matrix.csv',
    index=False,
    encoding='utf-8-sig'
)
data_qc.to_csv(PROCESS_DIR / '03_data_quality_checks.csv', index=False, encoding='utf-8-sig')
standardization_summary.to_csv(
    PROCESS_DIR / '04_standardization_summary.csv',
    index=False,
    encoding='utf-8-sig'
)


In [ ]:
parallel_summary, null_eigenvalues, n_retained = parallel_analysis(
    X,
    N_PERMUTATIONS,
    PARALLEL_PERCENTILE,
    PARALLEL_RANDOM_SEED
)
if n_retained < 1:
    raise ValueError('Parallel analysis did not retain any components.')
null_columns = [f'PC{index}' for index in range(1, null_eigenvalues.shape[1] + 1)]
null_iterations = pd.DataFrame(null_eigenvalues, columns=null_columns)
null_iterations.insert(0, 'Permutation', np.arange(1, N_PERMUTATIONS + 1))
null_iterations.to_csv(
    PROCESS_DIR / '05_parallel_analysis_permutation_eigenvalues.csv',
    index=False,
    encoding='utf-8-sig'
)
parallel_summary.to_csv(
    RESULTS_DIR / '01_parallel_analysis_summary.csv',
    index=False,
    encoding='utf-8-sig'
)
all_components = min(n_words, n_features)
model, scores, loadings, eigenvalues, ratios = fit_pca(X, all_components)
loadings, scores, sign_orientation = orient_by_largest_loading(loadings, scores, feature_names)
component_labels = [f'PC{index}' for index in range(1, all_components + 1)]
retained_labels = component_labels[:n_retained]
if not np.allclose(eigenvalues, parallel_summary['Observed Eigenvalue'], atol=1e-10):
    raise ValueError('PCA eigenvalues do not match the parallel-analysis eigenvalues.')
component_summary = pd.DataFrame(
    {
        'Component': component_labels,
        'Eigenvalue': eigenvalues,
        'Explained Variance Percentage': ratios * 100,
        'Cumulative Variance Percentage': np.cumsum(ratios) * 100,
        'Retained': [index < n_retained for index in range(all_components)]
    }
).merge(parallel_summary[['Component', 'Null 95th Percentile', 'Observed Minus Null 95th']], on='Component', how='left')
loadings_all = pd.DataFrame(loadings, index=feature_names, columns=component_labels)
loadings_all.index.name = 'Emotion Feature'
scores_all = pd.DataFrame(scores, index=emotion_words, columns=component_labels)
scores_all.index.name = 'Emotion Word'
loadings_retained = loadings_all[retained_labels].copy()
scores_retained = scores_all[retained_labels].copy()
top_loadings = make_top_loading_table(loadings_retained, N_TOP_LOADINGS)
extreme_words = make_extreme_word_table(scores_retained, N_EXTREME_WORDS)
component_summary.to_csv(
    RESULTS_DIR / '02_component_summary.csv',
    index=False,
    encoding='utf-8-sig'
)
sign_orientation.to_csv(RESULTS_DIR / '03_sign_orientation.csv', index=False, encoding='utf-8-sig')
loadings_all.to_csv(RESULTS_DIR / '04_component_loadings_all.csv', encoding='utf-8-sig')
loadings_retained.to_csv(RESULTS_DIR / '05_component_loadings_retained.csv', encoding='utf-8-sig')
scores_all.to_csv(RESULTS_DIR / '06_emotion_word_scores_all.csv', encoding='utf-8-sig')
scores_retained.to_csv(RESULTS_DIR / '07_emotion_word_scores_retained.csv', encoding='utf-8-sig')
top_loadings.to_csv(RESULTS_DIR / '08_top_loading_features.csv', index=False, encoding='utf-8-sig')
extreme_words.to_csv(
    RESULTS_DIR / '09_component_extreme_words.csv',
    index=False,
    encoding='utf-8-sig'
)


In [ ]:
summary_workbook = RESULTS_DIR / 'principal_component_analysis_summary.xlsx'
with pd.ExcelWriter(summary_workbook, engine='openpyxl') as writer:
    data_qc.to_excel(writer, sheet_name='Data_QC', index=False)
    standardization_summary.to_excel(writer, sheet_name='Standardization', index=False)
    parallel_summary.to_excel(writer, sheet_name='Parallel_Analysis', index=False)
    component_summary.to_excel(writer, sheet_name='Component_Summary', index=False)
    sign_orientation.to_excel(writer, sheet_name='Sign_Orientation', index=False)
    loadings_retained.reset_index().to_excel(writer, sheet_name='Component_Loadings', index=False)
    scores_retained.reset_index().to_excel(writer, sheet_name='Word_Scores', index=False)
    top_loadings.to_excel(writer, sheet_name='Top_Loadings', index=False)
    extreme_words.to_excel(writer, sheet_name='Extreme_Words', index=False)
